# 📦 P1 — Data Engineer (v3 — CORRIGÉ)
## Speech-to-Retrieval System (SRS) — INPT Project

### ✅ Changements v3
- ✅ Split corrigé : `train.100` et `train.360` (pas `train.clean.100`)
- ✅ Dataset : `openslr/librispeech_asr`, config `clean`
- ✅ Wikipedia : `wikimedia/wikipedia` version `20231101.en`
- ✅ Zéro `trust_remote_code`, zéro login requis


## ✅ CELLULE 1 — Installation

In [1]:
!pip install -q datasets torchcodec librosa soundfile transformers pandas tqdm scikit-learn wikipedia-api
print('✅ Installation terminée')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.8/47.8 kB 2.9 MB/s eta 0:00:00
✅ Installation terminée


## ✅ CELLULE 2 — Imports & Configuration

In [2]:
import os, re, random, warnings
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from pathlib import Path
from tqdm import tqdm
from collections import Counter
from datasets import load_dataset
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

CONFIG = {
    'sample_rate'       : 16000,
    'min_duration_sec'  : 1.0,
    'max_duration_sec'  : 20.0,
    'min_snr_db'        : 5.0,
    'target_audio_count': 5000,
    'chunk_size_tokens' : 512,
    'chunk_overlap'     : 50,
    'target_doc_count'  : 10000,
    'tokenizer_name'    : 'sentence-transformers/all-mpnet-base-v2',
    'audio_clean_dir'   : '../data/audio_clean/',
    'chunks_dir'        : '../data/chunks/',
    'output_dir'        : '../data/output/',
}

for p in [CONFIG['audio_clean_dir'], CONFIG['chunks_dir'], CONFIG['output_dir']]:
    os.makedirs(p, exist_ok=True)

print('⚙️  Configuration OK')

⚙️  Configuration OK


## ✅ CELLULE 3 — Collecte audio LibriSpeech

**Splits disponibles sur `openslr/librispeech_asr` config `clean` :**
```
train.100   → 100h  d'audio propre  ✅
train.360   → 360h  d'audio propre  ✅
validation  → données de validation
test        → données de test
```

In [3]:
def collect_librispeech(split: str, n: int, id_prefix: str) -> list:
    """
    Collecte n samples depuis un split LibriSpeech.

    Args:
        split     : 'train.100' ou 'train.360'
        n         : nombre de samples à collecter
        id_prefix : préfixe pour les IDs (ex: 'libri100')

    Returns:
        liste de dicts {id, audio, text, source, duration}
    """
    print(f'📥 LibriSpeech clean / {split} → {n} samples...')

    ds = load_dataset(
        'openslr/librispeech_asr',
        'clean',
        split=split,          # ← 'train.100' ou 'train.360'
        streaming=True
    )

    samples = []
    for i, s in enumerate(tqdm(ds, total=n, desc=f'  {split}')):
        if i >= n:
            break

        audio = np.array(s['audio']['array'], dtype=np.float32)
        sr    = s['audio']['sampling_rate']

        # Resample si nécessaire (LibriSpeech est déjà en 16kHz)
        if sr != CONFIG['sample_rate']:
            audio = librosa.resample(audio, orig_sr=sr, target_sr=CONFIG['sample_rate'])

        samples.append({
            'id'      : f'{id_prefix}_{i:05d}',
            'audio'   : audio,
            'text'    : s['text'],
            'source'  : f'librispeech_{split}',
            'duration': len(audio) / CONFIG['sample_rate'],
        })

    print(f'   ✅ {len(samples)} samples collectés')
    return samples


# ── Collecte depuis les deux splits ─────────────────────────
samples_100 = collect_librispeech('train.100', n=3000, id_prefix='libri100')
samples_360 = collect_librispeech('train.360', n=2000, id_prefix='libri360')

all_samples = samples_100 + samples_360
random.shuffle(all_samples)

print(f'\n📊 Total collecté : {len(all_samples)} samples')
print(f'   train.100 : {len(samples_100)}')
print(f'   train.360 : {len(samples_360)}')
print(f'   Durée est.: {sum(s["duration"] for s in all_samples)/3600:.1f}h')

📥 LibriSpeech clean / train.100 → 3000 samples...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

  train.100: 100%|██████████| 3000/3000 [00:48<00:00, 61.42it/s] 


   ✅ 3000 samples collectés
📥 LibriSpeech clean / train.360 → 2000 samples...


Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/48 [00:00<?, ?it/s]

  train.360: 100%|██████████| 2000/2000 [00:16<00:00, 118.09it/s]

   ✅ 2000 samples collectés

📊 Total collecté : 5000 samples
   train.100 : 3000
   train.360 : 2000
   Durée est.: 17.7h


## ✅ CELLULE 4 — Prétraitement & nettoyage audio

In [4]:
def compute_snr(waveform: np.ndarray) -> float:
    signal_power = np.mean(waveform ** 2) + 1e-10
    noise_floor  = np.percentile(np.abs(waveform), 10) ** 2 + 1e-10
    return 10 * np.log10(signal_power / noise_floor)


def preprocess_sample(s: dict, out_dir: str) -> dict | None:
    audio, dur = s['audio'], s['duration']

    if dur < CONFIG['min_duration_sec']:  return None
    if dur > CONFIG['max_duration_sec']:
        audio = audio[:int(CONFIG['max_duration_sec'] * CONFIG['sample_rate'])]
        dur   = CONFIG['max_duration_sec']

    snr = compute_snr(audio)
    if snr < CONFIG['min_snr_db']:  return None

    peak = np.abs(audio).max()
    if peak > 0: audio = audio / peak

    filepath = os.path.join(out_dir, f"{s['id']}.wav")
    sf.write(filepath, audio, CONFIG['sample_rate'])

    return {
        'audio_id' : s['id'],
        'filename' : f"{s['id']}.wav",
        'filepath' : filepath,
        'text'     : s['text'],
        'source'   : s['source'],
        'duration' : round(dur, 3),
        'snr_db'   : round(snr, 2),
    }


print('🔧 Prétraitement audio...')
clean_samples = []
n_short = n_noisy = 0

for s in tqdm(all_samples, desc='Preprocessing'):
    r = preprocess_sample(s, CONFIG['audio_clean_dir'])
    if r:  clean_samples.append(r)
    elif s['duration'] < CONFIG['min_duration_sec']: n_short += 1
    else: n_noisy += 1

clean_samples = clean_samples[:CONFIG['target_audio_count']]

manifest_df = pd.DataFrame(clean_samples)
manifest_df.to_csv(os.path.join(CONFIG['output_dir'], 'audio_manifest.csv'), index=False)

print(f'\n📊 Résultats prétraitement :')
print(f'   ✅ Acceptés     : {len(clean_samples)}')
print(f'   ❌ Trop courts  : {n_short}')
print(f'   ❌ Trop bruités : {n_noisy}')
print(f'\n💾 audio_manifest.csv sauvegardé')

🔧 Prétraitement audio...


Preprocessing: 100%|██████████| 5000/5000 [00:49<00:00, 100.68it/s]



📊 Résultats prétraitement :
   ✅ Acceptés     : 4985
   ❌ Trop courts  : 0
   ❌ Trop bruités : 15

💾 audio_manifest.csv sauvegardé


## ✅ CELLULE 5 — Corpus texte (Wikipedia via HuggingFace)

In [5]:
# ════════════════════════════════════════════════════════════
# CORPUS TEXTE — wikimedia/wikipedia
# Splits disponibles : train
# Config             : 20231101.en
# Accès              : public, sans login
# ════════════════════════════════════════════════════════════

print('📥 Chargement Wikipedia (wikimedia/wikipedia 20231101.en)...')

wiki_ds = load_dataset(
    'wikimedia/wikipedia',
    '20231101.en',
    split='train',
    streaming=True
)

# ~600 articles longs → ~10 000 chunks de 512 tokens
N_WIKI = 600
raw_docs = []

for i, article in enumerate(tqdm(wiki_ds, total=N_WIKI, desc='Wikipedia')):
    if i >= N_WIKI:
        break
    text = article['text'].strip()
    # Garder seulement les articles suffisamment longs
    if len(text.split()) < 150:
        continue
    raw_docs.append({
        'title'     : article['title'],
        'text'      : text,
        'word_count': len(text.split()),
        'source'    : 'wikipedia',
    })

print(f'✅ {len(raw_docs)} articles collectés')
print(f'   Mots moy./article : {np.mean([d["word_count"] for d in raw_docs]):.0f}')

📥 Chargement Wikipedia (wikimedia/wikipedia 20231101.en)...


README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Wikipedia: 100%|██████████| 600/600 [00:01<00:00, 446.16it/s]

✅ 570 articles collectés
   Mots moy./article : 3888


## ✅ CELLULE 6 — Chunking (512 tokens avec overlap)

In [6]:
print(f'⏳ Chargement tokenizer : {CONFIG["tokenizer_name"]}')
tokenizer = AutoTokenizer.from_pretrained(CONFIG['tokenizer_name'])
print(f'✅ Tokenizer prêt')


def chunk_document(doc: dict, doc_idx: int) -> list:
    tokens = tokenizer.encode(doc['text'], add_special_tokens=False)
    cs, co = CONFIG['chunk_size_tokens'], CONFIG['chunk_overlap']
    chunks, start, cidx = [], 0, 0

    while start < len(tokens):
        end  = min(start + cs, len(tokens))
        text = tokenizer.decode(tokens[start:end], skip_special_tokens=True)

        if len(text.strip()) > 50:
            cid = f'doc{doc_idx:05d}_chunk{cidx:03d}'
            chunks.append({
                'chunk_id'   : cid,
                'doc_idx'    : doc_idx,
                'chunk_idx'  : cidx,
                'title'      : doc['title'],
                'source'     : doc['source'],
                'text'       : text,
                'token_count': end - start,
            })
            # Sauvegarder le fichier .txt
            with open(os.path.join(CONFIG['chunks_dir'], f'{cid}.txt'), 'w', encoding='utf-8') as f:
                f.write(text)
            cidx += 1

        start += (cs - co)
        if end == len(tokens): break

    return chunks


print('\n✂️  Chunking en cours...')
all_chunks = []
for i, doc in enumerate(tqdm(raw_docs, desc='Chunking')):
    all_chunks.extend(chunk_document(doc, i))

chunks_df = pd.DataFrame(all_chunks)
if len(chunks_df) > CONFIG['target_doc_count']:
    chunks_df = chunks_df.sample(CONFIG['target_doc_count'], random_state=42).reset_index(drop=True)

chunks_df.to_csv(os.path.join(CONFIG['output_dir'], 'corpus_chunks.csv'), index=False)

print(f'\n📊 Chunking terminé :')
print(f'   Total chunks  : {len(all_chunks)}')
print(f'   Chunks gardés : {len(chunks_df)}')
print(f'   Tokens moy.   : {chunks_df["token_count"].mean():.0f}')
print(f'\n💾 corpus_chunks.csv sauvegardé')

⏳ Chargement tokenizer : sentence-transformers/all-mpnet-base-v2


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

✅ Tokenizer prêt

✂️  Chunking en cours...


Chunking: 100%|██████████| 570/570 [00:15<00:00, 35.98it/s]



📊 Chunking terminé :
   Total chunks  : 6744
   Chunks gardés : 6744
   Tokens moy.   : 493

💾 corpus_chunks.csv sauvegardé


## ✅ CELLULE 7 — Création des paires audio–document

In [7]:
STOPWORDS = {
    'the','a','an','is','are','was','were','be','been','have','has',
    'do','does','did','will','would','could','should','to','of','in',
    'for','on','with','at','by','from','and','but','or','this','that',
    'these','those','i','you','he','she','it','we','they','what',
    'which','who','how','all','some','more','other','each','both',
}

def keywords(text: str, n: int = 10) -> set:
    words = re.findall(r'[a-z]+', text.lower())
    words = [w for w in words if w not in STOPWORDS and len(w) > 3]
    return set(w for w, _ in Counter(words).most_common(n))

def jaccard(t1: str, t2: str) -> float:
    k1, k2 = keywords(t1), keywords(t2)
    if not k1 or not k2: return 0.0
    return len(k1 & k2) / len(k1 | k2)


print('🔗 Création des paires audio–document...')
sample_chunks = chunks_df.sample(min(1000, len(chunks_df)), random_state=42)
pairs = []

for audio in tqdm(clean_samples, desc='Matching'):
    best_score, best_chunk = -1.0, None
    for _, chunk in sample_chunks.iterrows():
        score = jaccard(audio['text'], chunk['text'])
        if score > best_score:
            best_score, best_chunk = score, chunk

    if best_chunk is not None:
        pairs.append({
            'audio_file'  : audio['filename'],
            'audio_path'  : audio['filepath'],
            'query_text'  : audio['text'],
            'chunk_id'    : best_chunk['chunk_id'],
            'doc_title'   : best_chunk['title'],
            'match_score' : round(best_score, 4),
            'source_audio': audio['source'],
        })

pairs_df = pd.DataFrame(pairs)
pairs_df.to_csv(os.path.join(CONFIG['output_dir'], 'pairs.csv'), index=False)

print(f'\n✅ {len(pairs_df)} paires créées')
print(f'   Score moyen : {pairs_df["match_score"].mean():.4f}')
pairs_df[['audio_file', 'chunk_id', 'match_score']].head()

🔗 Création des paires audio–document...


Matching: 100%|██████████| 4985/4985 [25:18<00:00,  3.28it/s]


✅ 4985 paires créées
   Score moyen : 0.0729


,audio_file,chunk_id,match_score
0,libri100_01383.wav,doc00009_chunk010,0.1111
1,libri100_01881.wav,doc00304_chunk003,0.1111
2,libri360_01901.wav,doc00472_chunk008,0.1111
3,libri100_01524.wav,doc00140_chunk030,0.0526
4,libri360_01654.wav,doc00260_chunk022,0.0769


## ✅ CELLULE 8 — Split train/val/test + rapport final

In [8]:
train_df, temp_df = train_test_split(pairs_df, test_size=0.20, random_state=42)
val_df,  test_df  = train_test_split(temp_df,  test_size=0.50, random_state=42)

for name, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    df.to_csv(os.path.join(CONFIG['output_dir'], f'pairs_{name}.csv'), index=False)

print('=' * 55)
print('📊 RAPPORT FINAL — P1 Data Engineer v3')
print('=' * 55)
print(f'\n🎙️  AUDIO')
print(f'   Fichiers .wav  : {len(clean_samples)}')
print(f'   Durée totale   : {sum(s["duration"] for s in clean_samples)/3600:.2f}h')
print(f'   Format         : WAV · 16 kHz · mono')
for src, cnt in manifest_df['source'].value_counts().items():
    print(f'   └─ {src:35s} : {cnt}')

print(f'\n📄  CORPUS TEXTE')
print(f'   Chunks (512 tok) : {len(chunks_df)}')
print(f'   Articles source  : {chunks_df["doc_idx"].nunique()}')
print(f'   Tokens moy/chunk : {chunks_df["token_count"].mean():.0f}')

print(f'\n🔗  PAIRES audio–document')
print(f'   Total  : {len(pairs_df)}')
print(f'   Train  : {len(train_df)}  (80%)')
print(f'   Val    : {len(val_df)}  (10%)')
print(f'   Test   : {len(test_df)}  (10%)')

print(f'\n📁  FICHIERS LIVRÉS')
for fname in ['audio_manifest.csv','corpus_chunks.csv','pairs.csv',
              'pairs_train.csv','pairs_val.csv','pairs_test.csv']:
    fpath = os.path.join(CONFIG['output_dir'], fname)
    size  = os.path.getsize(fpath)/1024 if os.path.exists(fpath) else 0
    status = '✅' if os.path.exists(fpath) else '⚠️ '
    print(f'   {status} {fname:30s} {size:8.1f} KB')

print('\n' + '=' * 55)
print('✅  Dataset P1 prêt — livrer à P2 et P3 !')
print('=' * 55)

📊 RAPPORT FINAL — P1 Data Engineer v3

🎙️  AUDIO
   Fichiers .wav  : 4985
   Durée totale   : 17.61h
   Format         : WAV · 16 kHz · mono
   └─ librispeech_train.100               : 3000
   └─ librispeech_train.360               : 1985

📄  CORPUS TEXTE
   Chunks (512 tok) : 6744
   Articles source  : 570
   Tokens moy/chunk : 493

🔗  PAIRES audio–document
   Total  : 4985
   Train  : 3988  (80%)
   Val    : 498  (10%)
   Test   : 499  (10%)

📁  FICHIERS LIVRÉS
   ✅ audio_manifest.csv               1445.3 KB
   ✅ corpus_chunks.csv               15719.5 KB
   ✅ pairs.csv                        1505.5 KB
   ✅ pairs_train.csv                  1201.9 KB
   ✅ pairs_val.csv                     152.4 KB
   ✅ pairs_test.csv                    151.4 KB

✅  Dataset P1 prêt — livrer à P2 et P3 !


In [11]:
import shutil
from google.colab import files
import os
import tempfile

# Define the output zip file name
zip_file_name = 'project_data_and_notebook.zip'
data_dir = './data'

print(f"Préparation de l'archive '{zip_file_name}'...")

# Manually set the notebook filename as 'google.colab.files' does not provide a method to get it programmatically.
# IMPORTANT: Please replace 'YOUR_NOTEBOOK_NAME.ipynb' with the actual name of your current Colab notebook file.
# For example, if your notebook is named 'P1_Data_Engineer.ipynb', change the line below to:
# notebook_filename = 'P1_Data_Engineer.ipynb'
notebook_filename = 'P1_Data_Engineer_v3.ipynb' # <--- IMPORTANT: Update this line with your notebook's name!
notebook_path = f"/content/{notebook_filename}"

# Create a temporary directory
with tempfile.TemporaryDirectory() as tmpdir:
    # Copy the 'data' directory into the temporary directory
    if os.path.exists(data_dir):
        shutil.copytree(data_dir, os.path.join(tmpdir, 'data'))
    else:
        print(f"Attention: Le dossier '{data_dir}' n'existe pas et ne sera pas inclus.")

    # Copy the notebook file into the temporary directory
    if os.path.exists(notebook_path):
        shutil.copy2(notebook_path, os.path.join(tmpdir, notebook_filename))
    else:
        print(f"Attention: Le fichier notebook '{notebook_filename}' n'a pas été trouvé et ne sera pas inclus. Veuillez vous assurer que le nom du notebook est correct.")

    # Create the zip archive from the temporary directory's contents
    shutil.make_archive(zip_file_name.replace('.zip', ''), 'zip', root_dir=tmpdir)

print(f"✅ Archive '{zip_file_name}' créée avec succès.")
print("Déclenchement du téléchargement...")

# Download the zip file
files.download(zip_file_name)

Préparation de l'archive 'project_data_and_notebook.zip'...
Attention: Le fichier notebook 'P1_Data_Engineer_v3.ipynb' n'a pas été trouvé et ne sera pas inclus. Veuillez vous assurer que le nom du notebook est correct.
✅ Archive 'project_data_and_notebook.zip' créée avec succès.
Déclenchement du téléchargement...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>